In [1]:
import google.cloud.bigquery as bq
import pandas_gbq
import pandas as pd
import numpy as np

In [2]:
project_id = 'mudah-analytics-222509'


In [3]:
 missing_ids = pd.read_gbq(
    ''' 
        with data_1 as 
      (       SELECT distinct DATE(A.paid_at) AS date,
                          A.pay_log_id AS pay_log_id
              FROM blocket.pay_log AS A
              WHERE  date(A.paid_at) ='2021-05-29'
              ORDER by date ASC 
      ),
      data_2 as  
      (        SELECT distinct DATE(B.paid_at) AS date,
                          B.pay_log_id AS pay_log_id
              FROM blocket_stream.pay_log AS B
              WHERE  Date(B.paid_at) ='2021-05-29'
            ORDER by date ASC
      ),
      data_3 as 
      (        SELECT distinct DATE(publish_time) AS date,
                              C.pay_log_id As pay_log_id, 
              FROM blocket_stream.pay_log_dbactions AS C
              WHERE date(C.publish_time) ='2021-05-29'
              ORDER by date ASC
      )

      select count(data_1.pay_log_id) as blocket_stream_missing, count(data_2.pay_log_id) as blocket_missing 
      from data_2
      full outer join data_1
      on data_2.pay_log_id =data_1.pay_log_id
      WHERE data_1.pay_log_id is null or data_2.pay_log_id is null
    ''', project_id = project_id)

In [4]:
missing_ids

,blocket_stream_missing,blocket_missing
0,6,358


In [5]:
left = missing_ids['blocket_missing'].dropna()
left

0    358
Name: blocket_missing, dtype: int64

In [6]:
right = missing_ids['blocket_stream_missing'].dropna()
right

0    6
Name: blocket_stream_missing, dtype: int64

In [7]:

deleted_check = pd.read_gbq(
    '''
    with data_1 as 
      (       SELECT distinct DATE(A.paid_at) AS date,
                          A.pay_log_id AS pay_log_id
              FROM blocket.pay_log AS A
              WHERE  date(A.paid_at) ='2021-05-29'
              ORDER by date ASC 
      ),
      data_2 as  
      (        SELECT distinct DATE(B.paid_at) AS date,
                          B.pay_log_id AS pay_log_id
              FROM blocket_stream.pay_log AS B
              WHERE  Date(B.paid_at) ='2021-05-29'
            ORDER by date ASC
      ),
      data_3 as 
      (        SELECT distinct DATE(publish_time) AS date,
                              C.pay_log_id As pay_log_id, 
              FROM blocket_stream.pay_log_dbactions AS C
              WHERE date(C.publish_time) ='2021-05-29'
              ORDER by date ASC
      ),
      missing_id_left as
      ( select  data_2.pay_log_id as blocket_missing 
          from data_1
          full outer join data_2
          on data_1.pay_log_id =data_2.pay_log_id 
          WHERE data_1.pay_log_id is null
      )    

      select count(missing_id_left.blocket_missing)  as check_left, count(data_3.pay_log_id) as data3_ID_exist
      from missing_id_left 
      left join data_3
      on missing_id_left.blocket_missing = data_3.pay_log_id  

       
    ''', project_id = project_id)


In [8]:
deleted_check

,check_left,data3_ID_exist
0,358,349


In [10]:

exist_check_date = pd.read_gbq(
  '''

with data_1 as 
      (       SELECT distinct DATE(A.paid_at) AS date,
                          A.pay_log_id AS pay_log_id
              FROM blocket.pay_log AS A
              WHERE  date(A.paid_at) ='2021-05-29'
              ORDER by date ASC 
      ),
      data_2 as  
      (        SELECT distinct DATE(B.paid_at) AS date,
                          B.pay_log_id AS pay_log_id
              FROM blocket_stream.pay_log AS B
              WHERE  Date(B.paid_at) ='2021-05-29'
            ORDER by date ASC
      ),
      data_3 as 
      (        SELECT distinct DATE(publish_time) AS date,
                              C.pay_log_id As pay_log_id, 
              FROM blocket_stream.pay_log_dbactions AS C
              WHERE date(C.publish_time) ='2021-05-29'
              ORDER by date ASC
      ),

      missing_id_left as
      ( select  data_2.pay_log_id as blocket_missing 
            from data_1
            full outer join data_2
            on data_1.pay_log_id =data_2.pay_log_id 
            WHERE data_1.pay_log_id is null 
      ),    

        missing_id_right as
            ( 
                select count(data_1.pay_log_id) as blocket_missing_stream
                from data_1
                full outer join data_2
                on data_1.pay_log_id =data_2.pay_log_id 
                WHERE  data_2.pay_log_id is null
            ),  
      deleted_check as
        (
        select missing_id_left.blocket_missing  as check_left, data_3.pay_log_id as data3_ID_exist
        from missing_id_left 
        left join data_3
        on missing_id_left.blocket_missing = data_3.pay_log_id  
        )

        select  missing_id_right.blocket_missing_stream as check_right, data_2.pay_log_id as data2_ID_exist   
        from missing_id_right
        left join data_2
        on missing_id_right.blocket_missing_stream  = data_2.pay_log_id 


    ''', project_id = project_id)


In [11]:
exist_check_date

,check_right,data2_ID_exist
0,6,NaN


In [12]:
print ("Id's in full outer join               : " ,missing_ids)
print ("Id's in blocket                       : " ,left)
print ("Id's in blocket_stream                : ", right)
print('-'*80)
print ("Id's are deleted                      : " ,deleted_check['data3_ID_exist'].count())
print ("Id's not found                        : ", exist_check_date['data2_ID_exist'].count())

Id's in full outer join               :     blocket_stream_missing  blocket_missing
0                       6              358
Id's in blocket                       :  0    358
Name: blocket_missing, dtype: int64
Id's in blocket_stream                :  0    6
Name: blocket_stream_missing, dtype: int64
--------------------------------------------------------------------------------
Id's not found                        :  0
